# Lab 5.3 &mdash; Parallel Execution, Reducers and Disagreement

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Reproduce the parallel bug that never raises &mdash; a whole specialist's work, gone
- Declare a reducer per key, and prove the findings survive
- Write the test that catches it, because no log line will
- Settle a disagreement by authority and provenance rather than by headcount

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Module 3's reducers, with consequences.** There a lost key was a puzzle.
> Here it is a compliance finding that never reached the recommendation.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists
# Deterministic stand-ins. Each takes the run state and returns a PARTIAL state -- exactly
# the LangGraph node shape from Module 3 -- and reports what it spent. No model is called,
# so a graph's structure AND its cost can be graded offline and exactly. The "Run it for
# real" cells put the sandbox model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

Two agents in the same wave both return `{"findings": [...]}`. Fold them in with a plain
`dict.update` and the second overwrites the first. Both calls succeeded, nothing raised, and one
specialist's entire contribution is gone.

The summary still reads perfectly &mdash; a summary of one finding reads exactly as well as a summary
of two. **Nothing in your logs will show this.** Only a test that counts.

## Section 1 &mdash; Watch it disappear

Run the two independent specialists on the same state, then merge them the obvious way.

In [ ]:
def fan_out(state: dict, agent_names) -> list:
    """Run several agents on the SAME input state. None of them sees the others' output."""
    return [(name, AGENTS[name](dict(state))) for name in agent_names]


def merge_naive(state: dict, partials: list) -> dict:
    """Last write wins. This is what you get when you do not declare a reducer."""
    out = dict(state)
    for _, partial in partials:
        out.update(partial)
    return out


def base_state(ref: str = "PMT-1005") -> dict:
    """Everything wave 1 established, ready for the parallel wave."""
    seed = {"ref": ref, "tokens": 0}
    first = agent_ledger(seed)
    return {**seed, **first}


def _demo():
    partials = fan_out(base_state(), ["policy", "sanctions"])
    returned = sum(len(p.get("findings") or []) for _, p in partials)
    merged = merge_naive(base_state(), partials)
    print(f"  the two specialists returned  {returned} findings")
    print(f"  the merged state contains     {len(merged.get('findings') or [])}")
    print(f"  errors raised                 0")
    for f in merged.get("findings") or []:
        print(f"    survivor: [{f['by']}] {f['claim'][:56]}")
guard(_demo)

In [ ]:
# --- Self-check: Section 1
def _partials():
    return fan_out(base_state(), ["policy", "sanctions"])

check("both specialists really did return a finding",
      lambda: sum(len(p.get("findings") or []) for _, p in _partials()) == 2)
check("neither of them raised",
      lambda: all(isinstance(p, dict) for _, p in _partials()))
check("but the naive merge keeps only one",
      lambda: len(merge_naive(base_state(), _partials()).get("findings") or []) == 1,
      "one specialist's entire contribution is gone, and nothing said so")
check("the survivor is whichever one ran last -- an ordering accident",
      lambda: merge_naive(base_state(), _partials())["findings"][0]["by"] == "sanctions")
check("the token count is wrong too, and in the cheaper direction",
      lambda: merge_naive(base_state(), _partials())["tokens"] == COST["sanctions"],
      "you will under-report your own spend, which is the one bug nobody reports")

## Section 2 &mdash; Declare a reducer per key

A reducer says how two writes to the same key combine. Lists append. Counters add. Booleans that
mean &ldquo;something is wrong&rdquo; OR together, so one blocker is enough to block.

In [ ]:
REDUCERS = {
    "findings":    lambda old, new: (old or []) + new,
    "problems":    lambda old, new: (old or []) + new,
    "tokens":      lambda old, new: (old or 0) + new,
    "blocked":     lambda old, new: bool(old) or bool(new),
    "needs_human": lambda old, new: bool(old) or bool(new),
}

def merge_reduced(state: dict, partials: list) -> dict:
    """Fold parallel results through the declared reducer for each key."""
    out = dict(state)
    for _, partial in partials:
        for key, value in partial.items():
            if key in REDUCERS:
                out[key] = REDUCERS[key](out.get(key), value)
            else:
                out[key] = value       # no reducer declared: last write still wins
    return out

In [ ]:
# --- Self-check: Section 2
def _reduced():
    return merge_reduced(base_state(), _partials())

check("both parallel findings survive, on top of what wave 1 established",
      lambda: len(_reduced()["findings"]) == 3)
check("so all three specialists are represented",
      lambda: {f["by"] for f in _reduced()["findings"]} == {"ledger", "policy", "sanctions"})
check("the spend is now the sum of all three, not whichever wrote last",
      lambda: _reduced()["tokens"] == COST["ledger"] + COST["policy"] + COST["sanctions"])
check("one blocker is enough to block",
      lambda: _reduced()["blocked"] is True)
check("merge order does not change the outcome",
      lambda: sorted(f["by"] for f in
                     merge_reduced(base_state(), list(reversed(_partials())))["findings"])
              == sorted(f["by"] for f in _reduced()["findings"]),
      "a reducer you can apply in either order is one that survives a scheduler you do not control")
check("a key with no declared reducer is still replaced",
      lambda: merge_reduced({"facts": {"a": 1}}, [("x", {"facts": {"b": 2}})])["facts"] == {"b": 2})
check("EVERY key the parallel specialists write has a reducer declared",
      lambda: {k for _, p in _partials() for k in p} <= set(REDUCERS),
      "this is the check to run in CI -- a new specialist writing a new key is the next silent loss")

## Section 3 &mdash; The test that catches it

No log line shows this. What shows it is asserting that everyone you dispatched came back.

In [ ]:
def contributed(state: dict) -> set:
    """Which specialists actually appear in the merged findings."""
    return {f["by"] for f in (state.get("findings") or [])}


def everyone_came_back(state: dict, dispatched) -> bool:
    """The assertion that catches a silent parallel loss: count what came back."""
    return set(dispatched) <= contributed(state)


def missing(state: dict, dispatched) -> set:
    """Who was dispatched and is not in the findings."""
    return set(dispatched) - contributed(state)

In [ ]:
# --- Self-check: Section 3
_dispatched = ["policy", "sanctions"]

check("the reduced merge passes the test",
      lambda: everyone_came_back(_reduced(), _dispatched) is True)
check("the naive merge FAILS it",
      lambda: everyone_came_back(merge_naive(base_state(), _partials()), _dispatched) is False,
      "this single assertion is the whole defence against the bug in Section 1")
check("and the failure names who went missing",
      lambda: missing(merge_naive(base_state(), _partials()), _dispatched) == {"policy"})
check("nothing is missing from a correct merge",
      lambda: missing(_reduced(), _dispatched) == set())
check("a specialist that returned no finding at all is also caught",
      lambda: everyone_came_back(_reduced(), ["policy", "sanctions", "writer"]) is False,
      "'it ran and found nothing' and 'its result was dropped' both need to surface")

## Section 4 &mdash; When they disagree

Three specialists say release. One, quoting the watchlist it read, says hold. Counting opinions
gets you the wrong answer confidently &mdash; which is Module 3's poisoning lab with a quorum.

In [ ]:
CONFLICT = [
    {"by": "writer",    "source": "inference", "verdict": "release",
     "claim": "nothing in the case looks unusual"},
    {"by": "policy",    "source": "policy",    "verdict": "release",
     "claim": "policy permits release once funded"},
    {"by": "ledger",    "source": "ledger",    "verdict": "release",
     "claim": "no block flag recorded against the payment"},
    {"by": "sanctions", "source": "watchlist", "verdict": "hold",
     "claim": "NORTHWIND is ON the watchlist"},
]

# Declared in advance, per question. On a sanctions question, compliance wins by definition.
AUTHORITY = {"sanctions": 3, "policy": 2, "ledger": 1, "writer": 0}

# Sources something other than a model can re-read.
CHECKABLE_SOURCES = {"ledger", "policy", "watchlist"}


def by_majority(findings: list) -> str:
    """The tempting rule. It counts opinions, and opinions are not evidence."""
    votes = {}
    for f in findings:
        votes[f["verdict"]] = votes.get(f["verdict"], 0) + 1
    return max(votes, key=votes.get)


def by_authority(findings: list, authority: dict = None) -> str:
    """The declared expert on this question wins, whatever the others think."""
    authority = AUTHORITY if authority is None else authority
    top = max(findings, key=lambda f: authority.get(f["by"], 0))
    return top["verdict"]


def checkable(finding: dict) -> bool:
    """Can this claim be settled by re-reading a source, rather than by asking again?"""
    return finding["source"] in CHECKABLE_SOURCES

In [ ]:
# --- Self-check: Section 4
check("three of the four say release",
      lambda: sum(1 for f in CONFLICT if f["verdict"] == "release") == 3)
check("so the majority rule releases a watchlisted payment",
      lambda: by_majority(CONFLICT) == "release",
      "confidently, unanimously among the three, and wrong")
check("authority holds it",
      lambda: by_authority(CONFLICT) == "hold")
check("the two rules disagree on this case",
      lambda: by_majority(CONFLICT) != by_authority(CONFLICT))
check("an agent with no declared authority ranks below every one that has it",
      lambda: by_authority(CONFLICT + [{"by": "stranger", "source": "inference",
                                        "verdict": "release", "claim": "looks fine"}]) == "hold")
check("authority is declared in advance, not derived from the case",
      lambda: set(AUTHORITY) >= {f["by"] for f in CONFLICT},
      "a rule chosen while looking at one disagreement is a rule fitted to that disagreement")
check("exactly one finding rests on nothing re-readable",
      lambda: [f["by"] for f in CONFLICT if not checkable(f)] == ["writer"])
check("and the dissenting finding is one of the checkable ones",
      lambda: checkable(next(f for f in CONFLICT if f["verdict"] == "hold")) is True,
      "which is why you can settle this by reading the watchlist rather than by taking a vote")

def _settle():
    print(f"  {'rule':14}{'verdict':10}")
    print("  " + "-" * 26)
    print(f"  {'majority':14}{by_majority(CONFLICT):10}")
    print(f"  {'authority':14}{by_authority(CONFLICT):10}")
    print()
    for f in sorted(CONFLICT, key=lambda f: -AUTHORITY.get(f["by"], 0)):
        mark = "checkable" if checkable(f) else "not checkable"
        print(f"  {f['by']:10} {f['verdict']:8} {mark:14} {f['claim'][:44]}")
guard(_settle)

## Run it for real

Hand the model the four findings and ask it to settle them. Then hand it the same four with the
sources removed. The question is whether provenance changes its answer &mdash; and whether you would
be willing to depend on that.

In [ ]:
if llm_ready():
    def _judge():
        def render(findings, with_sources):
            return "\n".join(
                (f"- [{f['by']}, source={f['source']}] {f['claim']} -> {f['verdict']}"
                 if with_sources else f"- {f['claim']} -> {f['verdict']}")
                for f in findings)
        for label, sourced in (("with sources   ", True), ("without sources", False)):
            reply = ask("Four agents disagree about whether one payment may be released. "
                        "Give the verdict and one sentence of reasoning.\n\n"
                        + render(CONFLICT, sourced))
            print(f"  [{label}] {reply.strip()[:200]}")
            print()
    guard(_judge)

### Read it

If removing the sources flips the answer to *release*, provenance did the work &mdash; and that is
good news, because provenance is something you control. If the model holds either way, do not
turn that into a control: `by_authority` is four lines and cannot be argued out of its answer.

**What you take from this lab:** declare a reducer for every key two branches can write, assert
that everyone you dispatched came back, and settle disagreements on authority and sources rather
than on a headcount.

In [ ]:
score()

## Your turn

1. Add a fifth specialist that writes a new key with no reducer. The CI check in Section 2 should
   fail. Make sure it does &mdash; that check is the only thing standing between you and the next
   silent loss.
2. `by_authority` breaks ties arbitrarily. Two equal-authority agents disagreeing is a real case:
   decide whether it escalates or falls back to provenance, and write it.
3. The `blocked` reducer ORs, so one blocker blocks. Build the opposite case &mdash; a key where OR is
   wrong &mdash; and say what that tells you about choosing reducers by data type.